In [1]:
from data_loader import get_mix_instruct
from utility_functions.delift_se import get_delift_se_utility
from utility_functions.encodes import get_encodes_utility
from subset import create_subset, get_subset
import random

prompts, references, ds_name = get_mix_instruct("train", 21000)
utility, utility_name = get_delift_se_utility(prompts, references, ds_name)
utility_enc, utility_name_enc = get_encodes_utility(prompts, references, ds_name)
subset, subset_name = create_subset(utility, utility_name, k =1)
subset_enc, subset_name_enc = create_subset(utility_enc, utility_name_enc, k =1)
s_prompts, s_references = get_subset(subset, prompts, references)

prompts_val, references_val, _ = get_mix_instruct("validation", 5000)

random.seed(42)  # Set seed for reproducibility
selected_indices = random.sample(range(len(prompts_val)), 50)
prompts_val = [prompts_val[i] for i in selected_indices]
references_val = [references_val[i] for i in selected_indices]

Dataset: mix-instruct_train_21000 found in cache, loading from cache ✅
Utility: mix-instruct_train_21000_delift-se found in cache, loading from cache ✅
Utility: mix-instruct_train_21000_encodes found in cache, loading from cache ✅
Subset: mix-instruct_train_21000_delift-se_1 found in cache, loading from cache ✅
Subset: mix-instruct_train_21000_encodes_1 found in cache, loading from cache ✅
Dataset: mix-instruct_validation_5000 found in cache, loading from cache ✅


In [ ]:
from scipy.stats import spearmanr
import numpy as np

x = np.array(subset_enc)[:, 0].astype(int)
y = x

correlation, p_value = spearmanr(x, y)
print(f"Spearman's Rank Correlation: {correlation}")

In [3]:
from sentence_transformers import InputExample, losses
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("BAAI/bge-base-en")
model.device

2025-03-30 22:13:32.383078: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-30 22:13:32.399107: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743372812.416094   24481 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743372812.421213   24481 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1743372812.434792   24481 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

device(type='cuda', index=0)

In [26]:
sentences = [f'{prompts_val[0]} {references_val[0]}', f'{prompts_val[2]} {references_val[2]}']
embeddings = model.encode(sentences)
similarity = util.cos_sim(embeddings[0], embeddings[1])

print("Cosine Similarity:", similarity.item())

Cosine Similarity: 0.36882901191711426


In [25]:
model_name = 'BAAI/bge-large-en-v1.5'
model = SentenceTransformer(model_name)

In [16]:
data_val = [f'{prompt} {reference}' for prompt, reference in zip(prompts_val, references_val)]

In [36]:
import random
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

# Load BGE model
model_name = 'BAAI/bge-large-en-v1.5'
model = SentenceTransformer(model_name)

# Prepare the data: Each sentence is treated as a separate example
train_examples = [InputExample(texts=[sentence, sentence]) for sentence in data_val]

# Dataloader
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=32)

# Use MultipleNegativesRankingLoss to push embeddings apart
train_loss = losses.MultipleNegativesRankingLoss(model)

# Fine-tune
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=20,
    warmup_steps=100,
    output_path="cache/models/bge-v1.5-finetuned"
)

Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.
Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


In [37]:
from sentence_transformers import util

model = SentenceTransformer("cache/models/bge-v1.5-finetuned")

# Encode sentences
embeddings = model.encode(data_val, convert_to_tensor=True)

# Compute pairwise cosine similarities
cosine_sim = util.pytorch_cos_sim(embeddings, embeddings)
print("Pairwise Cosine Similarity Matrix:\n", cosine_sim)

Pairwise Cosine Similarity Matrix:
 tensor([[1.0000, 0.2300, 0.2493,  ..., 0.2844, 0.4482, 0.1970],
        [0.2300, 1.0000, 0.2710,  ..., 0.2409, 0.4476, 0.2102],
        [0.2493, 0.2710, 1.0000,  ..., 0.3631, 0.2034, 0.2121],
        ...,
        [0.2844, 0.2409, 0.3631,  ..., 1.0000, 0.2126, 0.2471],
        [0.4482, 0.4476, 0.2034,  ..., 0.2126, 1.0000, 0.2213],
        [0.1970, 0.2102, 0.2121,  ..., 0.2471, 0.2213, 1.0000]],
       device='cuda:0')


In [29]:
from sentence_transformers import util

model = SentenceTransformer('BAAI/bge-large-en-v1.5')

# Encode sentences
embeddings = model.encode(data_val, convert_to_tensor=True)

# Compute pairwise cosine similarities
cosine_sim = util.pytorch_cos_sim(embeddings, embeddings)
print("Pairwise Cosine Similarity Matrix:\n", cosine_sim)

Pairwise Cosine Similarity Matrix:
 tensor([[1.0000, 0.3953, 0.3688,  ..., 0.3841, 0.5768, 0.3586],
        [0.3953, 1.0000, 0.4282,  ..., 0.3795, 0.5457, 0.4011],
        [0.3688, 0.4282, 1.0000,  ..., 0.4496, 0.3299, 0.3577],
        ...,
        [0.3841, 0.3795, 0.4496,  ..., 1.0000, 0.3147, 0.3624],
        [0.5768, 0.5457, 0.3299,  ..., 0.3147, 1.0000, 0.3724],
        [0.3586, 0.4011, 0.3577,  ..., 0.3624, 0.3724, 1.0000]],
       device='cuda:0')


In [14]:
train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=64)
train_loss = losses.MultipleNegativesRankingLoss(model)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=4,
    warmup_steps=100,
    show_progress_bar=True,
    output_path="cache/models/bge-finetuned"
)

Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.
Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss


KeyboardInterrupt: 

In [2]:
import importlib
import utility_functions.encodes
importlib.reload(utility_functions.encodes)
from utility_functions.encodes import get_encodes_utility
utility_enc2, utility_name_enc2 = get_encodes_utility(prompts, references, ds_name + 't1', embedding_model_name="cache/models/bge-finetuned")
subset_enc2, subset_name_enc2 = create_subset(utility_enc2, utility_name_enc2, k =1)

Utility: mix-instruct_train_21000t1_encodes found in cache, loading from cache ✅
Subset: mix-instruct_train_21000t1_encodes_1 found in cache, loading from cache ✅


In [3]:
from scipy.stats import spearmanr
import numpy as np

x = np.array(subset_enc)[:, 0].astype(int)
y = np.array(subset_enc2)[:, 0].astype(int)

correlation, p_value = spearmanr(x, y)
print(f"Spearman's Rank Correlation: {correlation}")

Spearman's Rank Correlation: 0.34989339449084994
